# 03 - Phan tich ket qua

Notebook nay tong hop cac ket qua experiment chinh cua bai toan Speech Emotion Recognition tren RAVDESS. Muc tieu la so sanh baseline, feature ablation, regularization va model cuoi cung dung cho demo Streamlit.

## 1. Setup

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path('..').resolve()
RESULTS_PATH = ROOT / 'experiments' / 'results.csv'
FIGURE_DIR = ROOT / 'report' / 'figures'
EXTERNAL_COMPARE_PATH = ROOT / 'checkpoints' / 'external_compare_old_models' / 'external_compare_summary.csv'

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_colwidth', 120)

## 2. Ket qua experiment chinh

In [ ]:
results = pd.read_csv(RESULTS_PATH)
metric_cols = ['test_loss', 'test_acc', 'test_f1_macro', 'test_f1_weight']

display(results[['run_name', 'model', 'feature_type', *metric_cols, 'notes']])

In [ ]:
plot_df = results.copy()
plot_df['run_label'] = plot_df['run_name'].replace({
    'cnn1d_mfcc_baseline_organic': 'Baseline\nMFCC + Adam',
    'cnn1d_mfcc_delta_fix1_organic': 'Delta\nAdam',
    'cnn1d_mfcc_delta_ls005_do025_adam': 'Delta + LS/DO\nAdam',
    'cnn1d_mfcc_delta_ls005_adamw': 'Final\nAdamW',
})

fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(plot_df))
width = 0.25

ax.bar([i - width for i in x], plot_df['test_acc'], width=width, label='Accuracy')
ax.bar(x, plot_df['test_f1_macro'], width=width, label='Macro F1')
ax.bar([i + width for i in x], plot_df['test_f1_weight'], width=width, label='Weighted F1')

ax.set_xticks(list(x))
ax.set_xticklabels(plot_df['run_label'])
ax.set_ylim(0, 1.0)
ax.set_ylabel('Score')
ax.set_title('RAVDESS test metrics by experiment')
ax.legend(loc='lower right')

for container in ax.containers:
    ax.bar_label(container, fmt='%.3f', fontsize=8, padding=2)

fig.tight_layout()
plt.show()

### Nhan xet nhanh

- Baseline `MFCC + Adam` dat ket qua kha tot va la moc so sanh chinh.
- Them `delta + delta-delta` rieng le khong cai thien, cho thay feature dong hoc co the lam bai toan kho hon neu chua regularize hop ly.
- Cau hinh `label smoothing + dropout` voi Adam van chua vuot baseline tren split hien tai.
- Model cuoi `AdamW + label smoothing + dropout + mfcc_delta` dat ket qua tot nhat sau khi re-evaluate checkpoint hien tai.

## 3. Muc cai thien so voi baseline

In [ ]:
baseline = results.loc[results['run_name'] == 'cnn1d_mfcc_baseline_organic'].iloc[0]
improvement = results[['run_name', 'test_acc', 'test_f1_macro', 'test_f1_weight']].copy()
improvement['delta_acc_vs_baseline'] = improvement['test_acc'] - baseline['test_acc']
improvement['delta_macro_f1_vs_baseline'] = improvement['test_f1_macro'] - baseline['test_f1_macro']
improvement['delta_weighted_f1_vs_baseline'] = improvement['test_f1_weight'] - baseline['test_f1_weight']

display(improvement)

## 4. Confusion matrix cua model cuoi

In [ ]:
cm_path = FIGURE_DIR / 'confusion_matrix_cnn1d_mfcc_delta_ls005_adamw.png'

if cm_path.exists():
    img = plt.imread(cm_path)
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.imshow(img)
    ax.axis('off')
    ax.set_title('Final model confusion matrix')
    plt.show()
else:
    print(f'Not found: {cm_path}')

## 5. External sanity check (khong phai benchmark chinh)

Plan cua project tap trung vao RAVDESS. Cac dataset nhu TESS, SAVEE va `emotion_audio` co domain va cach gan nhan khac nhau, nen ket qua ben duoi chi dung de kiem tra kha nang generalize so bo, khong nen xem la bang ket qua chinh.

In [ ]:
if EXTERNAL_COMPARE_PATH.exists():
    external = pd.read_csv(EXTERNAL_COMPARE_PATH)
    display(external)
else:
    external = None
    print(f'External comparison file not found: {EXTERNAL_COMPARE_PATH}')

In [ ]:
if external is not None:
    external_plot = external.copy()
    external_plot['run_label'] = external_plot['run_name'].replace({
        'cnn1d_mfcc_baseline_organic': 'Baseline',
        'cnn1d_mfcc_delta_fix1_organic': 'Delta',
        'cnn1d_mfcc_delta_ls005_do025_adam': 'Delta + Adam',
        'cnn1d_mfcc_delta_ls005_adamw': 'Final AdamW',
    })

    pivot = external_plot.pivot(index='dataset', columns='run_label', values='acc')
    display(pivot)

    ax = pivot.plot(kind='bar', figsize=(11, 5))
    ax.set_ylim(0, 1.0)
    ax.set_ylabel('Accuracy')
    ax.set_title('External sanity check accuracy')
    ax.legend(title='Model', bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.xticks(rotation=20, ha='right')
    plt.tight_layout()
    plt.show()

### Dien giai external check

- Final AdamW rat manh tren RAVDESS va co xu huong tot hon tren TESS.
- Ket qua tren SAVEE va `emotion_audio` thap, cho thay model van phu thuoc domain RAVDESS.
- Diem nay phu hop voi han che trong plan: giong that hoac dataset khac co the khac giong dien vien RAVDESS, dan den domain mismatch.

## 6. Ket luan cho report

Model duoc chon la `cnn1d_mfcc_delta_ls005_adamw`. Ket qua tot nhat khong den tu viec them delta rieng le, ma den tu to hop feature dong hoc, label smoothing, dropout phu hop va AdamW. Tuy nhien, cac external sanity check cho thay model chua generalize on dinh sang domain khac, nen huong phat trien tiep theo la fine-tune/augment voi du lieu da domain hon.